In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install gymnasium
!pip install stable-baselines3

In [ ]:
import gymnasium as gym

from gymnasium import spaces

import numpy as np

import matplotlib.pyplot as plt

from stable_baselines3 import PPO

import pandas as pd

In [ ]:
CALIFORNIA_TENSOR_PATH = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/california/grids/32x32/state_tensor.npy"

california_tensor = np.load(
    CALIFORNIA_TENSOR_PATH
).astype(np.float32)

print(california_tensor.shape)

In [ ]:
channel_names = [

    "Fire",
    "Fuel",
    "Wind X",
    "Wind Y",
    "Terrain",
    "Temperature",
    "Humidity"
]

fig, axes = plt.subplots(2,4, figsize=(15,8))

axes = axes.flatten()

for i in range(7):

    im = axes[i].imshow(
        california_tensor[i]
    )

    axes[i].set_title(channel_names[i])

    plt.colorbar(im, ax=axes[i])

axes[7].axis('off')

plt.tight_layout()

plt.show()

In [ ]:
class CaliforniaEvaluationEnv(gym.Env):

    def __init__(self):

        super(CaliforniaEvaluationEnv, self).__init__()

        self.grid_size = 32

        self.max_steps = 200

        self.current_step = 0

        self.initial_tensor = np.load(
            "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/california/grids/32x32/state_tensor.npy"
        ).astype(np.float32)

        self.state = self.initial_tensor.copy()

        self.action_space = spaces.Discrete(5)

        self.observation_space = spaces.Box(

            low=0.0,
            high=1.0,

            shape=(7,32,32),

            dtype=np.float32
        )

        self.agent_pos = [16,16]

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.state = self.initial_tensor.copy()

        self.current_step = 0

        self.agent_pos = [16,16]

        return self.state, {}

    def step(self, action):

        self.current_step += 1

        x, y = self.agent_pos

        # =====================
        # MOVEMENT
        # =====================

        if action == 0:
            x = max(0, x-1)

        elif action == 1:
            x = min(self.grid_size-1, x+1)

        elif action == 2:
            y = max(0, y-1)

        elif action == 3:
            y = min(self.grid_size-1, y+1)

        self.agent_pos = [x,y]

        # =====================
        # SUPPRESSION
        # =====================

        for dx in [-1,0,1]:

            for dy in [-1,0,1]:

                nx = x + dx
                ny = y + dy

                if (
                    0 <= nx < self.grid_size and
                    0 <= ny < self.grid_size
                ):

                    self.state[0, nx, ny] *= 0.2

        # =====================
        # FIRE SPREAD
        # =====================

        new_fire = self.state[0].copy()

        for i in range(1, self.grid_size-1):

            for j in range(1, self.grid_size-1):

                if self.state[0, i, j] > 0.2:

                    neighbors = [

                        (i-1,j),
                        (i+1,j),
                        (i,j-1),
                        (i,j+1)
                    ]

                    for ni, nj in neighbors:

                        fuel = self.state[1, ni, nj]

                        wind_factor = (
                            self.state[2, ni, nj] +
                            self.state[3, ni, nj]
                        ) / 2

                        terrain_factor = self.state[4, ni, nj]

                        spread_prob = (

                            0.01 +
                            0.15 * fuel +
                            0.08 * wind_factor +
                            0.08 * terrain_factor
                        )

                        if np.random.rand() < spread_prob:

                            new_fire[ni, nj] = min(
                                1.0,
                                new_fire[ni, nj] + 0.12
                            )

        self.state[0] = new_fire

        # =====================
        # FIRE DECAY
        # =====================

        self.state[0] *= 0.97

        self.state[0] = np.clip(
            self.state[0],
            0,
            1
        )

        self.state[0][self.state[0] < 0.02] = 0

        # =====================
        # FUEL DEPLETION
        # =====================

        self.state[1] -= self.state[0] * 0.003

        self.state[1] = np.clip(
            self.state[1],
            0,
            1
        )

        # =====================
        # REWARD
        # =====================

        total_fire = np.sum(self.state[0])

        suppression_bonus = 0

        if self.state[0, x, y] < 0.1:
            suppression_bonus = 2

        reward = -total_fire + suppression_bonus

        # =====================
        # TERMINATION
        # =====================

        done = False

        if self.current_step >= self.max_steps:
            done = True

        if total_fire < 0.1:
            done = True

        return self.state, reward, done, False, {}

In [ ]:
env = CaliforniaEvaluationEnv()

In [ ]:
SAUDI_MODEL_PATH = "/content/drive/MyDrive/PyroRL_Saudi_Project/models/ppo_saudi_32x32"

saudi_model = PPO.load(SAUDI_MODEL_PATH)

print("Saudi PPO loaded successfully.")

In [ ]:
CALIFORNIA_MODEL_PATH = "/content/drive/MyDrive/PyroRL_Saudi_Project/models/ppo_california_32x32"

california_model = PPO.load(
    CALIFORNIA_MODEL_PATH
)

print("California PPO loaded successfully.")

In [ ]:
def evaluate_model(

    model,
    env,
    episodes=20

):

    episode_rewards = []

    burned_cells = []

    fire_intensities = []

    for ep in range(episodes):

        obs, info = env.reset()

        done = False

        total_reward = 0

        while not done:

            action, _ = model.predict(
                obs,
                deterministic=True
            )

            obs, reward, done, truncated, info = env.step(action)

            total_reward += reward

        episode_rewards.append(total_reward)

        burned = np.sum(
            env.state[0] > 0.2
        )

        burned_cells.append(burned)

        fire_intensity = np.sum(env.state[0])

        fire_intensities.append(fire_intensity)

    return {

        "mean_reward": np.mean(episode_rewards),

        "std_reward": np.std(episode_rewards),

        "mean_burned_cells": np.mean(burned_cells),

        "mean_fire_intensity": np.mean(fire_intensities)
    }

In [ ]:
saudi_on_california = evaluate_model(

    saudi_model,
    env,
    episodes=20
)

print(saudi_on_california)

In [ ]:
california_on_california = evaluate_model(

    california_model,
    env,
    episodes=20
)

print(california_on_california)

In [ ]:
results_df = pd.DataFrame([

    {

        "Train Environment": "Saudi",

        "Test Environment": "California",

        "Mean Reward":
        saudi_on_california["mean_reward"],

        "Reward Std":
        saudi_on_california["std_reward"],

        "Burned Cells":
        saudi_on_california["mean_burned_cells"],

        "Fire Intensity":
        saudi_on_california["mean_fire_intensity"]
    },

    {

        "Train Environment": "California",

        "Test Environment": "California",

        "Mean Reward":
        california_on_california["mean_reward"],

        "Reward Std":
        california_on_california["std_reward"],

        "Burned Cells":
        california_on_california["mean_burned_cells"],

        "Fire Intensity":
        california_on_california["mean_fire_intensity"]
    }
])

results_df

In [ ]:
plt.figure(figsize=(8,5))

labels = [

    "Saudi → California",
    "California → California"
]

rewards = [

    saudi_on_california["mean_reward"],
    california_on_california["mean_reward"]
]

plt.bar(labels, rewards)

plt.ylabel("Mean Reward")

plt.title(
    "Zero-Shot Transfer Evaluation"
)

plt.grid(axis='y')

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

burned = [

    saudi_on_california["mean_burned_cells"],
    california_on_california["mean_burned_cells"]
]

plt.bar(labels, burned)

plt.ylabel("Burned Cells")

plt.title(
    "Wildfire Containment Comparison"
)

plt.grid(axis='y')

plt.show()

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/PyroRL_Saudi_Project/results"

import os

os.makedirs(SAVE_PATH, exist_ok=True)

results_df.to_csv(

    f"{SAVE_PATH}/zero_shot_transfer_results.csv",

    index=False
)

print("Results saved successfully.")

In [ ]:
print("""

==============================
ZERO-SHOT TRANSFER SUMMARY
==============================

This experiment evaluates whether a wildfire suppression policy trained
in Saudi Arabian wildfire conditions generalizes to California wildfire dynamics.

A significant performance drop indicates:

- ecological specialization
- environmental domain shift
- limited cross-regional generalization

The comparison against California-native PPO establishes
the transfer gap quantitatively.

==============================

""")